# Vedant Shirgaonkar
# D114 | D2-2

# Item-Based Collaborative Filtering 

In [1]:
import pandas as pd
import numpy as np

In [2]:
df_rat = pd.read_csv('ratings.csv')
df_mov = pd.read_csv('movies.csv')

In [3]:
df_rat.head()

,userId,movieId,rating,timestamp
0,1,1,4.0,964982703
1,1,3,4.0,964981247
2,1,6,4.0,964982224
3,1,47,5.0,964983815
4,1,50,5.0,964982931


In [4]:
df_mov.head()

,movieId,title,genres
0,1,Toy Story (1995),Adventure|Animation|Children|Comedy|Fantasy
1,2,Jumanji (1995),Adventure|Children|Fantasy
2,3,Grumpier Old Men (1995),Comedy|Romance
3,4,Waiting to Exhale (1995),Comedy|Drama|Romance
4,5,Father of the Bride Part II (1995),Comedy


In [5]:
# Merge ratings with movies
df = pd.merge(df_rat, df_mov, on='movieId')
df.head()

,userId,movieId,rating,timestamp,title,genres
0,1,1,4.0,964982703,Toy Story (1995),Adventure|Animation|Children|Comedy|Fantasy
1,1,3,4.0,964981247,Grumpier Old Men (1995),Comedy|Romance
2,1,6,4.0,964982224,Heat (1995),Action|Crime|Thriller
3,1,47,5.0,964983815,Seven (a.k.a. Se7en) (1995),Mystery|Thriller
4,1,50,5.0,964982931,"Usual Suspects, The (1995)",Crime|Mystery|Thriller


In [6]:
df.shape

(100836, 6)

In [7]:
df['title'].nunique()

9719

In [8]:
# Filtering Popular Movies: Set a threshold to filter out movies with very few ratings to ensure meaningful recommendations.
# Make a column for rating counts
rating_counts = df.groupby('title')['rating'].count().reset_index()
rating_counts.columns = ['title', 'ratingCount']
df = pd.merge(df, rating_counts, on='title')
df.head()

,userId,movieId,rating,timestamp,title,genres,ratingCount
0,1,1,4.0,964982703,Toy Story (1995),Adventure|Animation|Children|Comedy|Fantasy,215
1,1,3,4.0,964981247,Grumpier Old Men (1995),Comedy|Romance,52
2,1,6,4.0,964982224,Heat (1995),Action|Crime|Thriller,102
3,1,47,5.0,964983815,Seven (a.k.a. Se7en) (1995),Mystery|Thriller,203
4,1,50,5.0,964982931,"Usual Suspects, The (1995)",Crime|Mystery|Thriller,204


In [9]:
df.shape

(100836, 7)

In [10]:
min_ratings = 50
df = df[df['ratingCount'] >= min_ratings]
df.shape


(41362, 7)

In [14]:
df['title'].nunique()

450

In [11]:
# Construct a matrix where each row represents a movie, each column represents a user, and the values are the ratings given by users to movies.
rating_matrix = df.pivot_table(index='title', columns='userId', values='rating')
rating_matrix.fillna(0, inplace=True)
rating_matrix.head()

userId,1,2,3,4,5,6,7,8,9,10,...,601,602,603,604,605,606,607,608,609,610
title,,,,,,,,,,,,,,,,,,,,,
10 Things I Hate About You (1999),0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,3.0,0.0,5.0,0.0,0.0,0.0,0.0,0.0
12 Angry Men (1957),0.0,0.0,0.0,5.0,0.0,0.0,0.0,0.0,0.0,0.0,...,5.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
2001: A Space Odyssey (1968),0.0,0.0,0.0,0.0,0.0,0.0,4.0,0.0,0.0,0.0,...,0.0,0.0,5.0,0.0,0.0,5.0,0.0,3.0,0.0,4.5
28 Days Later (2002),0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,3.5,0.0,5.0
300 (2007),0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,3.0,...,0.0,0.0,0.0,0.0,3.0,0.0,0.0,5.0,0.0,4.0


In [12]:
# Use cosine similarity to measure how similar movies are based on user rating patterns.
from sklearn.metrics.pairwise import cosine_similarity
similarity_matrix = cosine_similarity(rating_matrix)
similarity_df = pd.DataFrame(similarity_matrix, index=rating_matrix.index, columns=rating_matrix.index)
similarity_df.head()

title,10 Things I Hate About You (1999),12 Angry Men (1957),2001: A Space Odyssey (1968),28 Days Later (2002),300 (2007),"40-Year-Old Virgin, The (2005)",A.I. Artificial Intelligence (2001),"Abyss, The (1989)",Ace Ventura: Pet Detective (1994),Ace Ventura: When Nature Calls (1995),...,Willy Wonka & the Chocolate Factory (1971),"Wizard of Oz, The (1939)","Wolf of Wall Street, The (2013)",X-Men (2000),X-Men: The Last Stand (2006),X2: X-Men United (2003),You've Got Mail (1998),Young Frankenstein (1974),Zombieland (2009),Zoolander (2001)
title,,,,,,,,,,,,,,,,,,,,,
10 Things I Hate About You (1999),1.000000,0.095586,0.191393,0.170720,0.351032,0.350903,0.221459,0.155244,0.229660,0.213071,...,0.280982,0.289917,0.133364,0.285599,0.215151,0.217504,0.305906,0.224106,0.227712,0.340590
12 Angry Men (1957),0.095586,1.000000,0.291756,0.217353,0.261876,0.236899,0.209119,0.162227,0.198630,0.186848,...,0.230947,0.324110,0.188794,0.237337,0.254711,0.235636,0.152301,0.220096,0.217742,0.196745
2001: A Space Odyssey (1968),0.191393,0.291756,1.000000,0.398424,0.339076,0.317919,0.460043,0.490033,0.271807,0.230101,...,0.301007,0.389963,0.191644,0.390909,0.340882,0.333269,0.260731,0.403752,0.288656,0.246335
28 Days Later (2002),0.170720,0.217353,0.398424,1.000000,0.423250,0.387243,0.437794,0.206805,0.206628,0.194713,...,0.267909,0.266093,0.228052,0.348975,0.403226,0.410175,0.197691,0.218362,0.436973,0.296377
300 (2007),0.351032,0.261876,0.339076,0.423250,1.000000,0.611255,0.394823,0.228963,0.361877,0.319655,...,0.292031,0.314623,0.295794,0.471374,0.544290,0.475067,0.292046,0.212877,0.480951,0.464538


In [13]:
similarity_df.shape

(450, 450)

In [ ]:
# Use k-Nearest Neighbors (k-NN) to find movies that are similar to a given movie.
from sklearn.neighbors import NearestNeighbors
model = NearestNeighbors(metric='cosine', algorithm='brute')
model.fit(rating_matrix)

NearestNeighbors(algorithm='brute', metric='cosine')

In [ ]:
# 5.	Finding Nearest Neighbors: Use k-Nearest Neighbors (k-NN) to find movies that are similar to a given movie.
def find_similar_movies(movie_title, n=5):
    movie_index = rating_matrix.index.get_loc(movie_title)
    distances, indices = model.kneighbors(rating_matrix.iloc[movie_index, :].values.reshape(1, -1), n_neighbors=n+1)
    similar_movies = rating_matrix.index[indices.flatten()[1:]]
    return similar_movies

In [18]:
# Example: Get top 5 similar movies to a given movie(User-input)
movie_to_check = input("Enter a movie title: ")
similar_movies = find_similar_movies(movie_to_check, n=5)
print(f"Movies similar to '{movie_to_check}':")
for movie in similar_movies:
    print(movie)

Movies similar to 'Toy Story (1995)':
Toy Story 2 (1999)
Jurassic Park (1993)
Independence Day (a.k.a. ID4) (1996)
Star Wars: Episode IV - A New Hope (1977)
Forrest Gump (1994)
